In [1]:
1

1

In [2]:
from __future__ import annotations

import json
import logging
import os
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI

In [3]:
load_dotenv("../.env", override=False)

True

In [4]:

TERMINAL_STATUSES = {"completed", "failed", "cancelled", "expired"}


def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def append_json_list(path: Path, item: Any) -> None:
    rows = []
    if path.is_file():
        existing = read_json(path)
        if isinstance(existing, list):
            rows = existing
    rows.append(item)
    write_json(path, rows)



def parse_custom_id(custom_id: str) -> tuple[str, str, int]:
    # New format: <paper_id>__<strategy>__turn_##.
    match_new = re.match(
        r"^(?P<paper>.+)__(?P<strategy>single-path|multi-path|update|verification)__turn_(?P<turn>\d+)$",
        custom_id,
    )
    if match_new:
        return match_new.group("paper"), match_new.group("strategy"), int(match_new.group("turn"))

    # Backward-compatible format: <paper_id>__turn_##.
    match_old = re.match(r"^(?P<paper>.+)__turn_(?P<turn>\d+)$", custom_id)
    if match_old:
        return match_old.group("paper"), "multi-path", int(match_old.group("turn"))

    raise ValueError(f"Invalid custom_id format: {custom_id}")


def deep_merge(base: Any, update: Any) -> Any:
    if update is None:
        return base
    if base is None:
        return update
    if isinstance(base, dict) and isinstance(update, dict):
        merged = dict(base)
        for k, v in update.items():
            merged[k] = deep_merge(merged.get(k), v)
        return merged
    return update


def extract_parsed_from_response_body(body: dict[str, Any]) -> dict[str, Any] | None:
    parsed = body.get("output_parsed")
    if isinstance(parsed, dict):
        return parsed

    output_text = body.get("output_text")
    if isinstance(output_text, str):
        text = output_text.strip()
        if text:
            try:
                obj = json.loads(text)
                if isinstance(obj, dict):
                    return obj
            except json.JSONDecodeError:
                pass

    output = body.get("output")
    if isinstance(output, list):
        for out_item in output:
            if not isinstance(out_item, dict):
                continue
            content = out_item.get("content")
            if not isinstance(content, list):
                continue
            for c in content:
                if not isinstance(c, dict):
                    continue
                txt = c.get("text")
                if isinstance(txt, str):
                    text = txt.strip()
                    if not text:
                        continue
                    try:
                        obj = json.loads(text)
                        if isinstance(obj, dict):
                            return obj
                    except json.JSONDecodeError:
                        continue
    return None


def read_bytes_from_file_content(content_obj: Any) -> bytes:
    if hasattr(content_obj, "read"):
        data = content_obj.read()
        if isinstance(data, bytes):
            return data
        if isinstance(data, str):
            return data.encode("utf-8")
    if hasattr(content_obj, "content"):
        data = content_obj.content
        if isinstance(data, bytes):
            return data
        if isinstance(data, str):
            return data.encode("utf-8")
    if isinstance(content_obj, bytes):
        return content_obj
    if isinstance(content_obj, str):
        return content_obj.encode("utf-8")
    raise TypeError("Unknown file content response type")


def collect_jobs_from_manifest(manifest_path: Path) -> list[dict[str, Any]]:
    rows = read_json(manifest_path)
    if not isinstance(rows, list):
        raise ValueError("manifest must be a JSON list")
    jobs = []
    for row in rows:
        if not isinstance(row, dict):
            continue
        if row.get("batch_id"):
            jobs.append(row)
    return jobs



In [5]:
def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()

In [6]:
"""
python assessment/03_batch_download.py --scenario-name s1_gpt54_single
"""

'\npython assessment/03_batch_download.py --scenario-name s1_gpt54_single\n'

In [7]:
manifest = Path("/work/hdd/bbkc/ma7/LiteratureReview/TileReviewCache/assessment/runs/s3_gpt5_multi/manifests/submit_manifest.json")
jobs = collect_jobs_from_manifest(manifest)

In [8]:
jobs

[{'saved_at_utc': '2026-04-15T14:35:32.674904+00:00',
  'scenario_name': 's3_gpt5_multi',
  'strategy': 'multi-path',
  'turn_index': 1,
  'focus_area': ['metadata', 'relevance'],
  'model': 'gpt-5-mini',
  'request_count': 50,
  'input_file_id': 'file-CPrNgvd1tjYhxFWDPmUSHp',
  'batch_id': 'batch_69dfa234289481908a5fa2ca1f90350b',
  'batch_status': 'validating',
  'request_jsonl': '/work/hdd/bbkc/ma7/LiteratureReview/TileReviewCache/assessment/runs/s3_gpt5_multi/batch/turn_01/requests.jsonl',
  'base_scenario': None,
  'base_results_dir': '/work/hdd/bbkc/ma7/LiteratureReview/TileReviewCache/assessment/runs/s3_gpt5_multi/results'},
 {'saved_at_utc': '2026-04-15T15:28:30.588169+00:00',
  'scenario_name': 's3_gpt5_multi',
  'strategy': 'multi-path',
  'turn_index': 2,
  'focus_area': ['context'],
  'model': 'gpt-5-mini',
  'request_count': 50,
  'input_file_id': 'file-1PHx6WqYJuMUQ45PdkH8de',
  'batch_id': 'batch_69dfae9e29c48190b269162295f857af',
  'batch_status': 'validating',
  'reque

In [16]:
client = OpenAI(api_key=api_key)
summary: dict[str, Any] = {"saved_at_utc": now_utc(), "scenario_name": 'debug', "jobs": []}

In [17]:
batch_id = str(jobs[0]["batch_id"])
batch_id

'batch_69ded8f1a4bc8190ace85b269c406d01'

In [27]:
batch_obj.model_dump()

{'id': 'batch_69ded8f1a4bc8190ace85b269c406d01',
 'completion_window': '24h',
 'created_at': 1776212209,
 'endpoint': '/v1/responses',
 'input_file_id': 'file-C58Ui8eVgncGrbvVxtSmnM',
 'object': 'batch',
 'status': 'completed',
 'cancelled_at': None,
 'cancelling_at': None,
 'completed_at': 1776212376,
 'error_file_id': 'file-352CWJebmmbCdrV1drccHz',
 'errors': None,
 'expired_at': None,
 'expires_at': 1776298609,
 'failed_at': None,
 'finalizing_at': 1776212371,
 'in_progress_at': 1776212272,
 'metadata': {'scenario_name': 's1_gpt54_single',
  'strategy': 'single-path',
  'turn_index': '1',
  'focus_area': '__ALL_FIELDS__',
  'model': 'gpt-5.4-mini'},
 'model': 'gpt-5.4-mini-2026-03-17',
 'output_file_id': None,
 'request_counts': {'completed': 0, 'failed': 50, 'total': 50},
 'usage': {'input_tokens': 0,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 0,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 0}}

In [19]:
batch_obj = client.batches.retrieve(batch_id)
status = getattr(batch_obj, "status", None)
snapshot = {
    "checked_at_utc": now_utc(),
    "batch_id": batch_id,
    "status": status,
    "output_file_id": getattr(batch_obj, "output_file_id", None),
    "error_file_id": getattr(batch_obj, "error_file_id", None),
    "request_counts": getattr(batch_obj, "request_counts", None),
}
snapshot

{'checked_at_utc': '2026-04-15T00:34:31.685091+00:00',
 'batch_id': 'batch_69ded8f1a4bc8190ace85b269c406d01',
 'status': 'completed',
 'output_file_id': None,
 'error_file_id': 'file-352CWJebmmbCdrV1drccHz',
 'request_counts': BatchRequestCounts(completed=0, failed=50, total=50)}

In [23]:
def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

dict

In [25]:
snapshot

{'checked_at_utc': '2026-04-15T00:34:31.685091+00:00',
 'batch_id': 'batch_69ded8f1a4bc8190ace85b269c406d01',
 'status': 'completed',
 'output_file_id': None,
 'error_file_id': 'file-352CWJebmmbCdrV1drccHz',
 'request_counts': BatchRequestCounts(completed=0, failed=50, total=50)}

In [24]:
append_json_list(Path("test/summary_batch_status_history.json"), snapshot)

TypeError: Object of type BatchRequestCounts is not JSON serializable

In [ ]:

        batch_obj = client.batches.retrieve(batch_id)
        status = getattr(batch_obj, "status", None)
        snapshot = {
            "checked_at_utc": now_utc(),
            "batch_id": batch_id,
            "status": status,
            "output_file_id": getattr(batch_obj, "output_file_id", None),
            "error_file_id": getattr(batch_obj, "error_file_id", None),
            "request_counts": getattr(batch_obj, "request_counts", None),
        }
        append_json_list("batch_status_history.json", snapshot)
        logger.info("batch status | batch_id=%s | status=%s", batch_id, status)
        if status in TERMINAL_STATUSES:
            break
        if time.time() - start > timeout_s:
            raise TimeoutError(f"Timed out waiting for batch {batch_id}")
        time.sleep(args.poll_interval_sec)

    status = getattr(batch_obj, "status", None)
    job_summary: dict[str, Any] = {
        "batch_id": batch_id,
        "status": status,
        "turn_index": turn_index,
        "output_file_id": getattr(batch_obj, "output_file_id", None),
        "error_file_id": getattr(batch_obj, "error_file_id", None),
    }

    if turn_index is None:
        turn_dir = scenario_root / "batch" / f"batch_{batch_id}"
    else:
        turn_dir = scenario_root / "batch" / f"turn_{turn_index:02d}"
    turn_dir.mkdir(parents=True, exist_ok=True)

    if status != "completed":
        write_json(turn_dir / "batch_status.json", job_summary)
        summary["jobs"].append(job_summary)
        continue

    output_file_id = getattr(batch_obj, "output_file_id", None)
    error_file_id = getattr(batch_obj, "error_file_id", None)
    output_path = turn_dir / "output.jsonl"
    error_path = turn_dir / "error.jsonl"

    if output_file_id and (args.force or not output_path.is_file()):
        content = client.files.content(output_file_id)
        output_path.write_bytes(read_bytes_from_file_content(content))
        logger.info("downloaded output file | batch_id=%s", batch_id)
    if error_file_id and (args.force or not error_path.is_file()):
        content = client.files.content(error_file_id)
        error_path.write_bytes(read_bytes_from_file_content(content))
        logger.info("downloaded error file | batch_id=%s", batch_id)

    parsed_dir = turn_dir / "parsed"
    parsed_dir.mkdir(parents=True, exist_ok=True)

    parsed_count = 0
    failed_count = 0
    if output_path.is_file():
        with output_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                custom_id = row.get("custom_id")
                if not isinstance(custom_id, str):
                    continue
                write_json(parsed_dir / f"{custom_id}.json", row)

                try:
                    paper_id, strategy, custom_turn = parse_custom_id(custom_id)
                except ValueError:
                    failed_count += 1
                    continue
                response = row.get("response", {})
                body = response.get("body", {}) if isinstance(response, dict) else {}
                if not isinstance(body, dict):
                    body = {}
                parsed = extract_parsed_from_response_body(body)
                if parsed is None:
                    failed_count += 1
                    continue

                results_paper_dir = scenario_root / "results" / paper_id
                results_paper_dir.mkdir(parents=True, exist_ok=True)
                write_json(results_paper_dir / f"turn_{custom_turn:02d}_raw.json", row)
                write_json(results_paper_dir / f"turn_{custom_turn:02d}_parsed.json", parsed)

                # Verification strategy returns patch suggestions and should not
                # auto-merge into extraction results.
                if strategy == "verification":
                    write_json(results_paper_dir / f"turn_{custom_turn:02d}_patch_response.json", parsed)
                else:
                    merged_path = results_paper_dir / "merged_extraction.json"
                    merged = read_json(merged_path) if merged_path.is_file() else {}
                    if not isinstance(merged, dict):
                        merged = {}
                    merged = deep_merge(merged, parsed)
                    write_json(merged_path, merged)
                parsed_count += 1

    job_summary.update(
        {
            "parsed_count": parsed_count,
            "parse_failed_count": failed_count,
            "output_path": str(output_path) if output_path.is_file() else None,
            "error_path": str(error_path) if error_path.is_file() else None,
        }
    )
    write_json(turn_dir / "batch_status.json", job_summary)
    append_json_list(scenario_root / "manifests" / "download_manifest.json", job_summary)
    summary["jobs"].append(job_summary)

write_json(scenario_root / "summary" / "download_summary.json", summary)
logger.info("download finished | scenario=%s", args.scenario_name)
